## Silver
### Consolidação, Padronização e Qualidade dos Dados

*Case Técnico — Engenharia de Dados · iFood*

---

**Objetivo**

Implementar a camada Silver do Lakehouse, responsável pela consolidação dos dados provenientes da camada Bronze, padronização dos schemas, validação da qualidade dos dados e tratamento das principais inconsistências identificadas, preparando o conjunto de dados para consumo analítico na camada Gold

**Fonte dos dados**

| | |
|---|---|
| Dataset | NYC TLC Trip Record Data |
| Período | Janeiro a Maio de 2023 |
| Serviços | Yellow Cab, Green Cab |
| Formato | Parquet(camada Bronze) |

**Atividades realizadas**

- Leitura consolidada dos dados da camada Bronze;
- Padronização dos schemas entre Yellow e Green Taxi;
- Validação dos tipos de dados;
- Auditoria de consistência dos atributos;
- Tratamento de inconsistências identificadas;
- Validação de registros duplicados;
- Preparação do conjunto de dados para a camada Gold.



#### **1. Instalando e carregando os pacotes**  

In [0]:
from pyspark.sql import functions as F

#### **2. Leitura dos arquivos**  
- Leitura dos 10 arquivos;
- Seleção dos arquivos de entrada;
- Cast dos atributos com datatype divergente, identificados na camada bronze, preservando a semântica e adotando tipo proposto na origem.
   - Foi feita a análise na camada Bronze, baseado em min e max, e concluímos que não há prejuízo aos dados em preservar o novo datatype proposto na origem.
- Padronização do nome das colunas.

In [0]:
BRONZE_PATH = "/Volumes/ifood_case/bronze/raw"

#### Yellow

In [0]:
yellow_files = sorted([
    f.path
    for f in dbutils.fs.ls(BRONZE_PATH)
    if f.path.endswith(".parquet")
    and "yellow_tripdata" in f.path
])

print(f"Arquivos Yellow encontrados: {len(yellow_files)}")

for file in yellow_files:
    print(file)

In [0]:
def ler_yellow_silver(path):
    df = spark.read.parquet(path)

    if "Airport_fee" in df.columns:
        df = df.withColumnRenamed("Airport_fee", "airport_fee")

    return (
        df
        .withColumn("VendorID", F.col("VendorID").cast("int"))
        .withColumn("passenger_count", F.col("passenger_count").cast("long"))
        .withColumn("RatecodeID", F.col("RatecodeID").cast("long"))
        .withColumn("payment_type", F.col("payment_type").cast("long"))
        .withColumn("airport_fee", F.col("airport_fee").cast("double"))
        .withColumn("ehail_fee", F.lit(0.0).cast("double"))
        .withColumn("trip_type", F.lit(0).cast("long"))
        .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")
        .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
        .withColumn("tipo", F.lit("yellow"))
    )

In [0]:
yellow_dfs = [ler_yellow_silver(path) for path in yellow_files]

df_yellow_silver = yellow_dfs[0]

for df in yellow_dfs[1:]:
    df_yellow_silver = df_yellow_silver.unionByName(df)

In [0]:
# Databricks não permite utilização de cache na versão serverless
# df_yellow_silver = df_yellow_silver.cache()

In [0]:
df_yellow_silver.printSchema()

In [0]:
df_yellow_silver.count()

#### Green

In [0]:
green_files = sorted([
    f.path
    for f in dbutils.fs.ls(BRONZE_PATH)
    if f.path.endswith(".parquet")
    and "green_tripdata" in f.path
])

print(f"Arquivos Green encontrados: {len(green_files)}")

for file in green_files:
    print(file)

In [0]:
def ler_green_silver(path):
    periodo = path.split("_2023-")[1].replace(".parquet", "")

    return (
        spark.read.parquet(path)
        .withColumn("VendorID", F.col("VendorID").cast("int"))
        .withColumn("passenger_count", F.col("passenger_count").cast("long"))
        .withColumn("RatecodeID", F.col("RatecodeID").cast("long"))
        .withColumn("payment_type", F.col("payment_type").cast("long"))
        .withColumn("trip_type", F.col("trip_type").cast("long"))
        .withColumn("airport_fee", F.lit(0.0).cast("double"))
        .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime")
        .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
        .withColumn("tipo", F.lit("green"))
    )    

In [0]:
green_dfs = [ler_green_silver(path) for path in green_files]

df_green_silver = green_dfs[0]

for df in green_dfs[1:]:
    df_green_silver = df_green_silver.unionByName(df)

In [0]:
# Databricks não permite utilização de cache na versão serverless
# df_green_silver = df_green_silver.cache()

In [0]:
df_green_silver.printSchema()

In [0]:
df_green_silver.count()

In [0]:
df_green_silver.show(10)

#### **3. Tratamento Inicial dos Dados** 

#### Union Yellow + Green

In [0]:
df_silver_bruto = df_yellow_silver.unionByName(df_green_silver)  # antes do cast final

print(f"Yellow: {df_yellow_silver.count():,} registros")
print(f"Green : {df_green_silver.count():,} registros")
print(f"Silver: {df_silver_bruto.count():,} registros")


In [0]:
# Databricks não permite utilização de cache na versão serverless
# df_yellow_silver.unpersist()
# df_green_silver.unpersist()

#### Cast das colunas

In [0]:
SCHEMA_SILVER = {
    "VendorID":              "int",
    "pickup_datetime":       "timestamp_ntz",
    "dropoff_datetime":      "timestamp_ntz",
    "store_and_fwd_flag":    "string",
    "RatecodeID":            "long",
    "PULocationID":          "long",
    "DOLocationID":          "long",
    "passenger_count":       "long",
    "trip_distance":         "double",
    "fare_amount":           "double",
    "extra":                 "double",
    "mta_tax":               "double",
    "tip_amount":            "double",
    "tolls_amount":          "double",
    "ehail_fee":             "double",
    "improvement_surcharge": "double",
    "total_amount":          "double",
    "payment_type":          "long",
    "trip_type":             "long",
    "congestion_surcharge":  "double",
    "airport_fee":           "double",
    "tipo":                  "string",
}

In [0]:
def aplicar_schema(df, schema_dict):
    faltantes = [c for c in schema_dict if c not in df.columns]
    if faltantes:
        raise ValueError(f"Colunas ausentes no df de origem: {faltantes}")

    return df.select([
        F.col(c).cast(t).alias(c) for c, t in schema_dict.items()
    ])

In [0]:
def validar_casts(df_raw, df_casted, schema_dict):
    exprs_raw = [F.count(c).alias(c) for c in schema_dict if c in df_raw.columns]
    exprs_cast = [F.count(c).alias(c) for c in schema_dict]

    raw_row = df_raw.select(*exprs_raw).collect()[0].asDict()
    cast_row = df_casted.select(*exprs_cast).collect()[0].asDict()

    resultado = []
    for c in schema_dict:
        antes = raw_row.get(c)
        depois = cast_row[c]
        if antes is None:
            continue
        perdidos = antes - depois
        resultado.append((c, antes, depois, perdidos))
        if perdidos > 0:
            print(f"⚠️  {c}: {perdidos} valores perdidos no cast ({antes} → {depois})")
        else:
            print(f"✅ {c}: nenhuma perda ({antes} não-nulos)")

    return spark.createDataFrame(resultado, ["coluna", "raw_nao_nulos", "cast_nao_nulos", "perdidos"])

In [0]:

df_silver = (
    aplicar_schema(df_silver_bruto, SCHEMA_SILVER)
    .withColumn("data_corrida", F.to_date("pickup_datetime"))
    .withColumn("ano_mes", F.date_format("pickup_datetime", "yyyy-MM"))
)

df_validacao = validar_casts(df_silver_bruto, df_silver, SCHEMA_SILVER)
display(df_validacao.filter(F.col("perdidos") > 0))

#### Deduplicação
Os arquivos do TLC Trip Record Data não possuem uma chave primária natural. Para avaliar a existência de registros duplicados, foram realizadas duas abordagens de validação:

- Duplicidade exata: considerando todos os atributos do conjunto de dados, não foram identificados registros duplicados.
- Duplicidade por chave lógica: considerando os atributos VendorID, pickup_datetime, dropoff_datetime, PULocationID, DOLocationID, passenger_count e total_amount, foram identificadas apenas 2 ocorrências.

Como não há evidências suficientes de que essas ocorrências representem registros efetivamente duplicados — podendo corresponder a corridas distintas com características operacionais e financeiras idênticas — optou-se por não realizar a deduplicação automática, preservando a integridade dos dados da camada Silver.


In [0]:
distintos = df_silver.dropDuplicates().count()

print(f"Registros distintos: {distintos}")

In [0]:
chave = ["VendorID", "pickup_datetime", "dropoff_datetime", "PULocationID", "DOLocationID", "passenger_count", "total_amount"]

dup_chave = (
    df_silver.groupBy(chave)
    .count()
    .filter("count > 1")
)

display(dup_chave) 

#### **4. Análise Exploratória e Validação da Qualidade dos Dados**



#### VendorID

Não foi identificado nenhum problema nos dados do atributo.

In [0]:
display(
    df_silver
    .groupBy("VendorID")
    .agg(F.count("*").alias("total_registros"))
    .orderBy("VendorID")
)

#### **passenger_count**

Foram identificados 451.561 registros (2,73% do total) com valores nulos no atributo passenger_count. Durante a análise exploratória, observou-se que os atributos RatecodeID e store_and_fwd_flag apresentam o mesmo padrão de nulidade nesses registros.

O atributo store_and_fwd_flag indica se a corrida foi armazenada temporariamente no veículo antes de ser transmitida ao provedor. A ocorrência simultânea de valores nulos nesses três atributos apenas sugere que parte dos metadados da corrida não foi registrada ou disponibilizada na origem dos dados.   

Essa hipótese não pode ser confirmada apenas com as informações presentes na base.

In [0]:
display(
    df_silver
    .groupBy("passenger_count")
    .count()
    .orderBy("passenger_count")
)

In [0]:
display(
    df_silver.filter(F.col("passenger_count").isNull())
      .agg(
          F.count("*").alias("total"),
          F.count("VendorID").alias("VendorID_preenchido"),
          F.count("total_amount").alias("total_amount_preenchido"),
          F.count("pickup_datetime").alias("pickup_preenchido"),
          F.count("dropoff_datetime").alias("dropoff_preenchido"),
          F.count("RatecodeID").alias("RatecodeID"),
          F.count("store_and_fwd_flag").alias("store_and_fwd_flag")
      )
)   

In [0]:
display(
    df_silver.agg(
        F.avg("passenger_count").alias("media"),
        F.expr("percentile_approx(passenger_count, 0.5)").alias("mediana")
    )
)

**_Tratamento adotado:_**     
Como os valores nulos representam apenas 2,73% do total de registros, optei por imputar a mediana da coluna passenger_count, preservando a natureza da variável e minimizando o impacto sobre as análises. A mediana foi escolhida por ser menos sensível a valores extremos.


In [0]:

mediana = df_silver.approxQuantile("passenger_count", [0.5], 0.01)[0]

df_silver = df_silver.fillna({"passenger_count": mediana})

In [0]:
display(
    df_silver.filter(F.col("passenger_count").isNull())
      .agg(
          F.count("*").alias("total"),
          F.count("VendorID").alias("VendorID_preenchido"),
          F.count("total_amount").alias("total_amount_preenchido"),
          F.count("pickup_datetime").alias("pickup_preenchido"),
          F.count("dropoff_datetime").alias("dropoff_preenchido"),
          F.count("RatecodeID").alias("RatecodeID"),
          F.count("store_and_fwd_flag").alias("store_and_fwd_flag")
      )
)   

**_Resultado:_**    
Após a substituição dos valores nulos de **passenger_count**, a mediana da variável permaneceu igual a 1, enquanto a média passou de 1,3603 para 1,3504. Essa pequena variação indica que o tratamento teve baixo impacto na distribuição da variável, reforçando a adequação do tratamento adotado para este caso.

In [0]:
display(
    df_silver.agg(
        F.avg("passenger_count").alias("media"),
        F.expr("percentile_approx(passenger_count, 0.5)").alias("mediana")
    )
)

#### **pickup_datetime** e **dropoff_datetime**
Não foram encontrados valores nulos nessas colunas, o que garante que não há timestamps em formato inválido, uma vez que ambos os atributos passaram por conversão (cast) no início do processo.         
Foram identificados registros com inconsistência temporal, em que dropoff_datetime é anterior a 
pickup_datetime.

In [0]:
df_timestamp_validacao = (
    df_silver
    .withColumn(
        "problema_timestamp",
        F.when(F.col("pickup_datetime").isNull(), "pickup_nulo")
         .when(F.col("dropoff_datetime").isNull(), "dropoff_nulo")
         .when(
             F.col("dropoff_datetime") < F.col("pickup_datetime"),
             "dropoff_menor_que_pickup"
         )
         .otherwise("valido")
    )
)

display(
    df_timestamp_validacao
    .filter(F.col("problema_timestamp") != "valido")
    .groupBy("ano_mes", "problema_timestamp")
    .agg(F.count("*").alias("total_registros"))
    .orderBy("ano_mes", "problema_timestamp")
)

- 93,96% (747/795) têm store_and_fwd_flag nulo.
- 6,03% (48/795) têm store_and_fwd_flag preenchido.   

Observou-se que cerca de 94% dos registros com inconsistência temporal também apresentam store_and_fwd_flag nulo. Embora exista uma forte associação entre as ocorrências, ela não é suficiente para concluir que ambas têm a mesma causa.

In [0]:
display(
    df_silver
    .filter(F.col("dropoff_datetime") < F.col("pickup_datetime"))
    .agg(
        F.count("*").alias("total_inconsistentes"),
        F.count("store_and_fwd_flag").alias("store_and_fwd_flag_preenchido")
    )
)

A análise da diferença entre os timestamps mostrou que 763 registros (96,0%) apresentam diferença de até 1 minuto, enquanto apenas 2 registros possuem diferença superior a 1 hora. Considerando que  representam um percentual irrelevante da base e que a grande maioria apresenta diferenças muito pequenas, optei por inverter os valores dos timestamps e seguir com o processamento.

In [0]:
display(
    df_silver
    .filter(F.col("dropoff_datetime") < F.col("pickup_datetime"))
    .withColumn(
        "dif_min",
        (
            F.unix_timestamp("pickup_datetime") -
            F.unix_timestamp("dropoff_datetime")
        ) / 60
    )
    .select(
        F.sum((F.col("dif_min") <= 1).cast("int")).alias("até_1_min"),
        F.sum(((F.col("dif_min") > 1) & (F.col("dif_min") <= 5)).cast("int")).alias("1_a_5_min"),
        F.sum(((F.col("dif_min") > 5) & (F.col("dif_min") <= 60)).cast("int")).alias("5_a_60_min"),
        F.sum((F.col("dif_min") > 60).cast("int")).alias("acima_60_min")
    )
)

**_Tratamento adotado:_**     
Foram criados dois novos atributos, pickup_datetime_tratado e dropoff_datetime_tratado, com os valores corrigidos. As colunas originais (pickup_datetime, dropoff_datetime) foram mantidas na camada Silver — não por serem usadas nas análises daqui em diante, mas para preservar o dado de origem e permitir auditoria caso um problema semelhante seja identificado futuramente.

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "pickup_datetime_tratado",
        F.least("pickup_datetime", "dropoff_datetime")
    )
    .withColumn(
        "dropoff_datetime_tratado",
        F.greatest("pickup_datetime", "dropoff_datetime")
    )
)

Validação dos registros remanescentes

In [0]:
display(
    df_silver
    .filter(F.col("dropoff_datetime_tratado") < F.col("pickup_datetime_tratado"))
    .count()
)

Quantidade de registros ajustados

In [0]:
display(
    df_silver.agg(
        F.sum(
            (
                F.col("pickup_datetime") !=
                F.col("pickup_datetime_tratado")
            ).cast("int")
        ).alias("pickup_alterado"),

        F.sum(
            (
                F.col("dropoff_datetime") !=
                F.col("dropoff_datetime_tratado")
            ).cast("int")
        ).alias("dropoff_alterado")
    )
)

#### **data_corrida**

Foram identificados 113 registros (0,0007% do total) com data_corrida fora do intervalo esperado pelo case (janeiro a maio de 2023), incluindo datas de anos anteriores e posteriores ao período da coleta.

In [0]:
display(
    df_silver
    .withColumn(
        "fora_do_periodo",
        (F.col("data_corrida") < "2023-01-01") | (F.col("data_corrida") > "2023-05-31")
    )
    .groupBy("fora_do_periodo")
    .agg(F.count("*").alias("total"))
)    

In [0]:
display(
    df_silver
    .filter((F.col("data_corrida") < "2023-01-01") | (F.col("data_corrida") > "2023-05-31"))
    .groupBy("tipo", "ano_mes")
    .agg(F.count("*").alias("total"))
    .orderBy("ano_mes")
)

**_Tratamento adotado:_**     
Diferente dos demais casos analisados nesta seção, não há explicação de negócio plausível para essas datas. Por representarem um volume irrelevante
e estarem fora do escopo definido pelo case, optou-se por remover esses registros da camada Silver.

In [0]:
df_silver = df_silver.filter(
    (F.col("data_corrida") >= "2023-01-01") & (F.col("data_corrida") <= "2023-05-31")
)

In [0]:
df_silver.count()


#### **total_amount**

O atributo total_amount foi validado por meio da recomposição dos seus componentes financeiros (fare_amount, extra, mta_tax, tip_amount, tolls_amount, improvement_surcharge, congestion_surcharge e airport_fee, quando aplicável).    

As seções a seguir apresentam a análise das situações inicialmente identificadas como potenciais inconsistências, a avaliação realizada em cada caso e a decisão adotada.

In [0]:
display(
    df_silver.select("total_amount").summary()
)

**_Conclusão do tratamento de total_amount:_**

Foram realizadas validações cruzadas envolvendo padrões por VendorID, payment_type, duração da corrida e a soma dos componentes de cobrança. Com base nesses achados, outliers explicados pela duração da corrida, valores negativos associados a uma regra de negócio específica, e apenas 218 registros sem explicação identificada,  concluiu-se que não havia necessidade de qualquer tratamento específico sobre o atributo total_amount.

In [0]:
q = (
    df_silver
    .selectExpr(
        "percentile_approx(total_amount, 0.25) as q1",
        "percentile_approx(total_amount, 0.75) as q3"
    )
    .first()
)

q1 = q["q1"]
q3 = q["q3"]
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

print(limite_inferior)
print(limite_superior)

In [0]:
display(
    df_silver
    .withColumn(
        "outlier",
        (F.col("total_amount") < limite_inferior) |
        (F.col("total_amount") > limite_superior)
    )
    .groupBy("outlier")
    .count()
)

In [0]:
display(
    df_silver
    .filter(
        (F.col("total_amount") < limite_inferior) |
        (F.col("total_amount") > limite_superior)
    )
    .select(
        "tipo",
        "ano_mes",
        "pickup_datetime",
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "tip_amount",
        "tolls_amount",
        "airport_fee",
        "total_amount"
    )
    .orderBy(F.desc("total_amount"))
)

A tabela a seguir se refere ao valor cobrado por tempo da corrida, para identificar se os valores mais altos são justificados por corridas mais longas.

In [0]:
# Tempo
df_tempos = (
    df_silver
    .withColumn(
        "duracao_minutos",
        (
            F.unix_timestamp("dropoff_datetime_tratado") -
            F.unix_timestamp("pickup_datetime_tratado")
        ) / 60.0
    )
    .filter(F.col("duracao_minutos") > 0)
)

A tabela abaixo detalha a distribuição de outliers dentro de cada faixa de duração.
 Nas faixas mais curtas (<10min e 10-20min), o valor médio dos outliers é aproximadamente o dobro do valor médio da faixa — comportamento compatível com valores de fato extremos. 
Nas faixas a partir de 30min, a proporção de outliers cresce significativamente (71,3% na faixa 30-60min, 89,3% em >60min), mas o valor médio desse grupo fica muito próximo do valor médio da faixa inteira — ou seja, o "outlier" praticamente coincide com o padrão da faixa.

Isso ocorre porque o limite do IQR foi calculado sobre a base inteira, dominada predominantemente por corridas curtas (faixas <10min e 10-20min somam mais de 75% dos registros), sem considerar que o valor da corrida cresce naturalmente com a duração. Como resultado, o critério global superestima a quantidade de outliers nas faixas de maior duração, classificando como anômalo o que sugere, na prática, o comportamento esperado de uma corrida mais longa.



In [0]:
# Agrupar por faixa
# Estatísticas gerais da faixa + estatísticas apenas dos outliers
df_faixas = (
    df_tempos
    .withColumn(
        "faixa_duracao",
        F.when(F.col("duracao_minutos") < 10, "<10 min")
         .when(F.col("duracao_minutos") < 20, "10-20 min")
         .when(F.col("duracao_minutos") < 30, "20-30 min")
         .when(F.col("duracao_minutos") < 60, "30-60 min")
         .otherwise(">60 min")
    )
    .withColumn(
        "outlier",
        (F.col("total_amount") < limite_inferior) |
        (F.col("total_amount") > limite_superior)
    )
)

display(
    df_faixas
    .groupBy("faixa_duracao")
    .agg(
        F.count("*").alias("corridas"),
        F.avg("duracao_minutos").alias("duracao_media"),
        F.avg("total_amount").alias("total_medio"),
        F.max("total_amount").alias("maior_total"),
        F.sum(F.col("outlier").cast("int")).alias("qtd_outliers"),
        F.avg(F.when(F.col("outlier"), F.col("total_amount"))).alias("total_medio_outlier"),
        F.expr("percentile(CASE WHEN outlier THEN total_amount END, 0.5)").alias("total_mediano_outlier")
    )
    .orderBy("duracao_media")
)

_**Valores negativos**_

In [0]:
display(
    df_silver.filter(F.col("total_amount") < 0)
        .agg(
            F.count("*").alias("total_negativos"),
            F.min("total_amount").alias("valor_minimo"),
            F.max("total_amount").alias("valor_maximo")
        )
)

A tabela abaixo tem por objetivo avaliar se há padrão de valores negativos por período, o que poderia ser justificado por problema pontual de ingestão.

In [0]:
display(
    df_silver
    .filter(F.col("total_amount") < 0)
    .groupBy(
        F.date_format("pickup_datetime", "yyyy-MM").alias("mes")
    )
    .count()
    .orderBy("mes")
)

A tabela a seguir detalha a ocorrência dos valores negativos por data completa, permitindo identificar se há concentração em dias específicos que sugira falha pontual de ingestão.

In [0]:
display(
    df_silver
    .filter(F.col("total_amount") < 0)
    .groupBy("data_corrida")
    .count()
    .orderBy("data_corrida")
)

Todos os registros com **total_amount** negativo pertencem ao VendorID = 2, o que sugere um comportamento relacionado às regras de negócio desse provedor.

Foi verificado que o **total_amount**, seja positivo ou negativo, corresponde exatamente ao somatório dos componentes da cobrança, não sendo encontradas evidências de inconsistência que justifiquem qualquer tratamento.

In [0]:
display(
    df_silver
    .groupBy("VendorID")
    .agg(
        F.sum((F.col("total_amount") < 0).cast("int")).alias("total_negativos")
    )
)

A análise não identificou associação entre valores negativos de total_amount e o atributo payment_type

In [0]:
display(
    df_silver
    .filter(F.col("total_amount") < 0)
    .groupBy("payment_type")
    .count()
    .orderBy("payment_type")
)

Sumariza os componentes do **total_amount** para validar se o valor total corresponde à soma de seus atributos financeiros, avaliando a possibilidade de inconsistências de cálculo.

In [0]:
display(
    df_silver
    .filter(F.col("total_amount") < 0)
    .withColumn(
        "total_calculado",
        F.coalesce(F.col("fare_amount"), F.lit(0.0)) +
        F.coalesce(F.col("extra"), F.lit(0.0)) +
        F.coalesce(F.col("mta_tax"), F.lit(0.0)) +
        F.coalesce(F.col("tip_amount"), F.lit(0.0)) +
        F.coalesce(F.col("tolls_amount"), F.lit(0.0)) +
        F.coalesce(F.col("improvement_surcharge"), F.lit(0.0)) +
        F.coalesce(F.col("congestion_surcharge"), F.lit(0.0)) +
        F.coalesce(F.col("Airport_fee"), F.col("airport_fee"), F.lit(0.0))
    )
    .withColumn(
        "validacao",
        F.when(
            F.abs(F.col("total_amount") - F.col("total_calculado")) <= 0.01,
            "Sem diferença"
        ).otherwise("Com diferença")
    )
    .groupBy("validacao")
    .count()
)

Os 218 registros abaixo apresentam diferença entre o somatório dos componentes financeiros e o valor registrado em total_amount. A inspeção individual foi realizada para identificar possíveis causas das divergências.

In [0]:
display(
    df_silver
    .filter(F.col("total_amount") < 0)
    .withColumn(
        "total_calculado",
        F.coalesce(F.col("fare_amount"), F.lit(0.0)) +
        F.coalesce(F.col("extra"), F.lit(0.0)) +
        F.coalesce(F.col("mta_tax"), F.lit(0.0)) +
        F.coalesce(F.col("tip_amount"), F.lit(0.0)) +
        F.coalesce(F.col("tolls_amount"), F.lit(0.0)) +
        F.coalesce(F.col("improvement_surcharge"), F.lit(0.0)) +
        F.coalesce(F.col("congestion_surcharge"), F.lit(0.0)) +
        F.coalesce(F.col("airport_fee"), F.lit(0.0))
    )
    .withColumn(
        "diferenca",
        F.round(F.col("total_amount") - F.col("total_calculado"), 4)
    )
    .filter(F.abs(F.col("diferenca")) > 0.01)
    .select(
        "ano_mes",
        "pickup_datetime",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "congestion_surcharge",
        "airport_fee",
        "total_amount",
        "total_calculado",
        "diferenca"
    )
)

#### **5. Grava Camada Silver**

In [0]:
(
    df_silver
    .repartition("ano_mes")
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("ano_mes")
    .saveAsTable("ifood_case.silver.trips")
)

In [0]:
%sql
-- Otimização física dentro de cada partição, para filtros por data completa
OPTIMIZE ifood_case.silver.trips
ZORDER BY (data_corrida)

In [0]:
%sql
-- Documentação da tabela
COMMENT ON TABLE ifood_case.silver.trips IS
'Camada Silver consolidando Yellow e Green Cab (NYC TLC, jan-mai/2023). Schema completo,
tipado e validado, deduplicado (chave de negócio + registro completo), com tratamento de
nulos em passenger_count e correção de timestamps invertidos. Mantém todas as colunas de
origem para suportar análises exploratórias adicionais; o corte para as colunas exigidas
pelo case (VendorID, passenger_count, total_amount, tpep_pickup_datetime,
tpep_dropoff_datetime) é aplicado na camada Gold.';

ALTER TABLE ifood_case.silver.trips ALTER COLUMN total_amount COMMENT
'Outliers (critério IQR) mantidos: justificados por corridas de maior duração. Valores
negativos mantidos: associados a regra de negócio específica do VendorID = 2.';

ALTER TABLE ifood_case.silver.trips ALTER COLUMN passenger_count COMMENT
'Nulos de origem (2,73% dos registros) imputados pela mediana.';